# AI Citizen Services RAG

## Libraries

In [1]:
!pip install -q langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters
!pip install -q transformers==4.52.4
!pip install -q sentence-transformers faiss-cpu accelerate
!pip install -q requests beautifulsoup4
!pip install -q fastapi uvicorn pyngrok

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import time
from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import threading
from flask import Flask, jsonify, request
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

/tmp/ipykernel_337/1926097700.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## LLM Model

In [3]:
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF")
except Exception:
    hf_token = os.environ.get("HF")

if not hf_token:
    raise RuntimeError("HF_TOKEN not found. Add it to Kaggle Secrets or set the HF_TOKEN env var.")

login(token=hf_token)

In [4]:
model_name = "UBC-NLP/NileChat-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.80G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [5]:
_eos = {tokenizer.eos_token_id}
_g = model.generation_config.eos_token_id
_eos.update(_g if isinstance(_g, (list, tuple)) else [_g])
_eos.discard(None)
EOS_IDS = sorted(_eos)
print("EOS ids:", EOS_IDS)


def generate_text(prompt, max_new_tokens=300):
    if tokenizer.chat_template:
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        add_special = False
    else:
        text = prompt
        add_special = True

    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=add_special).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            eos_token_id=EOS_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


EOS ids: [151643, 151645]


## Web Scrabing

In [6]:
# SESSION
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ar,en-US;q=0.9,en;q=0.8"
}

session = requests.Session()
session.headers.update(headers)

### KHADAMAT MISR

In [7]:
KHADAMAT_BASE_URL = "https://www.khadamatmisr.gov.eg"
KHADAMAT_SERVICES_URL = f"{KHADAMAT_BASE_URL}/serviceslist"

def get_khadamat_service_links():

    response = session.get(
        KHADAMAT_SERVICES_URL,
        timeout=30
    )

    print("Khadamat Misr status:", response.status_code)

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all("a", href=True):

        href = a["href"]
        name = a.get_text(" ", strip=True)

        if not name:
            continue

        full_url = urljoin(
            KHADAMAT_BASE_URL,
            href
        )

        if "/node/" in full_url:

            links.append({
                "name": name,
                "url": full_url
            })

    unique_links = {}

    for item in links:
        unique_links[item["url"]] = item

    return list(unique_links.values())

In [8]:
def clean_khadamat_text(text):

    text = re.sub(r"\s+", " ", text).strip()

    # أجزاء الموقع التي لا نحتاجها
    unwanted_sections = [
        "عن خدمات مصر",
        "كلمة السيد الوزير",
        "كلمة الدكتور أحمد رستم",
        "الخدمات دليل الخدمات",
        "الدعم و المساعدة",
        "الاسئلة الشائعة",
        "الشكاوى",
        "مواقع ذات صلة",
        "جميع الحقوق محفوظة",
        "مراكز خدمات مصر"
    ]

    for section in unwanted_sections:

        if section in text:

            text = text.split(
                section,
                1
            )[0]

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [9]:
_INVISIBLE = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]")


def build_khadamat_text(service_name, text):
    text = _INVISIBLE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    m = re.search(
        r"قنوات الخدمة و مراكز\W*خدمات مصر\W*المستندات المطلوبة\s*(.*?)\s*مراكز\W*خدمات مصر",
        text,
    )
    if not m or not m.group(1).strip():
        return None
    docs = [d.strip() for d in re.split(r"[•●▪]", m.group(1)) if d.strip()]
    body = "\n".join(f"- {d}" for d in docs)
    return f"اسم الخدمة: {service_name}\nالمستندات المطلوبة:\n{body}"


def scrape_khadamat_service(item):
    url = item["url"]
    try:
        response = session.get(url, timeout=30)

        if response.status_code != 200:
            print("Failed:", response.status_code, url)
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.find("title")
        if title:
            service_name = title.get_text(" ", strip=True).replace("| مركز خدمات مصر", "").strip()
        else:
            service_name = item["name"]

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        # التعديل الوحيد: التنضيف الجديد أولًا، والقديم كـ fallback
        clean_text = build_khadamat_text(service_name, text) or clean_khadamat_text(text)

        if not clean_text:
            return None

        return {
            "service_name": service_name,
            "url": url,
            "text": text,
            "clean_text": clean_text,
            "source": "خدمات مصر",
            "category": "Government Services",
        }

    except Exception as e:
        print("Error:", url, e)
        return None

In [10]:
EXCLUDED_KHADAMAT = {
    "عن خدمات مصر",
    "اتصل بنا",
    "كلمة الدكتور أحمد رستم – وزير التخطيط والتنمية الاقتصادية"
}

In [11]:
def scrape_khadamat_all(
    links,
    max_workers=10
):

    documents = []

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {

            executor.submit(
                scrape_khadamat_service,
                item
            ): item

            for item in links
        }


        for i, future in enumerate(
            as_completed(futures),
            start=1
        ):

            result = future.result()

            if result:

                # استبعاد الصفحات العامة
                if (
                    result["service_name"].strip()
                    not in EXCLUDED_KHADAMAT
                ):

                    documents.append(result)


            if i % 25 == 0:

                print(
                    f"Khadamat: "
                    f"{i}/{len(links)}"
                )


    return documents

### MINISTRY OF SUPPLY

In [12]:
MSIT_SERVICES = [
    {
        "service_name": "التقديم على منفذ تموينى",
        "url": "https://www.msit.gov.eg/?p=5066"
    },
    {
        "service_name": "التقديم على مخبز تمويني",
        "url": "https://www.msit.gov.eg/?p=5063"
    },
    {
        "service_name": "أماكن تنشيط البطاقات الذكية",
        "url": "https://www.msit.gov.eg/?p=5052"
    },
    {
        "service_name": "دورة استخراج بدل فاقد للبطاقة الذكية",
        "url": "https://www.msit.gov.eg/?p=5042"
    },
    {
        "service_name": "دورة استخراج بطاقة تموينية ذكية جديدة",
        "url": "https://www.msit.gov.eg/?p=5039"
    }
]

In [13]:
def scrape_msit_all():

    documents = []

    for i, item in enumerate(
        MSIT_SERVICES,
        start=1
    ):

        print(
            f"MSIT: {item['service_name']}"
        )

        result = scrape_msit_service(item)

        if result:
            documents.append(result)

        else:
            print(
                f"Skipped: {item['service_name']}"
            )

    return documents

In [14]:
def scrape_msit_service(item):
    try:

        response = session.get(
            item["url"],
            timeout=5
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "form",
            "aside"
        ]):
            tag.decompose()

        text = soup.get_text(
            " ",
            strip=True
        )

        text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()

        service_name = item["service_name"]

        share_marker = "Share:"
        share_pos = text.find(share_marker)

        if share_pos != -1:

            content = text[
                share_pos + len(share_marker):
            ].strip()

            arabic_share = "شارك هذا الموضوع"

            end_pos = content.find(arabic_share)

            if end_pos != -1:
                content = content[:end_pos].strip()

        else:
            print(
                "Skipped - content marker not found:",
                service_name
            )
            return None

        content = re.sub(
            r"\s+",
            " ",
            content
        ).strip()

        if len(content) < 20:
            print(
                "Skipped - content too short:",
                service_name
            )
            return None

        final_text = (
            f"اسم الخدمة: {service_name}\n\n"
            f"محتوى الخدمة:\n{content}"
        )

        return {
            "service_name": service_name,
            "url": item["url"],
            "text": content,
            "clean_text": final_text,
            "source": "وزارة التموين والتجارة الداخلية",
            "category": "Government Services"
        }

    except requests.exceptions.RequestException as e:

        print(
            "Skipped - request failed:",
            item["service_name"]
        )

        return None

    except Exception as e:

        print(
            "MSIT Error:",
            item["service_name"],
            e
        )

        return None

### PASSPORTS / IMMIGRATION

In [15]:
MOI_BASE_URL = (
    "https://enationality.moi.gov.eg"
)

MOI_SERVICES_URL = (
    "https://enationality.moi.gov.eg/"
    "UserApplications/Services/Index"
)

In [16]:
def get_moi_service_links():

    response = session.get(
        MOI_SERVICES_URL,
        timeout=30
    )

    print(
        "MOI status:",
        response.status_code
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all(
        "a",
        href=True
    ):

        name = a.get_text(
            " ",
            strip=True
        )

        href = a["href"]

        if not name:
            continue

        full_url = urljoin(
            MOI_BASE_URL,
            href
        )

        # نستبعد الروابط العامة
        if (
            "Services" in full_url
            or "Application" in full_url
            or "UserApplications" in full_url
        ):

            links.append({
                "name": name,
                "url": full_url
            })

    unique_links = {}

    for item in links:

        unique_links[item["url"]] = item

    return list(
        unique_links.values()
    )

In [17]:
def scrape_moi_service(service):

    try:

        category_id = service["service_url"].split(
            "CategoryId="
        )[1]

        url = (
            MOI_BASE_URL +
            "/UserApplications/Services/InstructionsData"
        )

        response = session.get(
            url,
            params={
                "CategoryId": category_id
            },
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        sections = {}

        for section in soup.select("section > div.mt-15"):

            title_tag = section.find("h2")

            if not title_tag:
                continue

            title = title_tag.get_text(
                " ",
                strip=True
            )

            content = section.find("p")

            if not content:
                continue

            # استخراج النص من li لو موجود
            items = content.find_all("li")

            if items:

                text = "\n".join(
                    item.get_text(
                        " ",
                        strip=True
                    )
                    for item in items
                )

            else:

                text = content.get_text(
                    " ",
                    strip=True
                )

            sections[title] = text

        service_name = service["service_name"]

        # بناء النص النهائي للـ RAG
        clean_text = f"""
اسم الخدمة: {service_name}

وصف الخدمة:
{service["description"]}

إرشادات الطلب:
{sections.get("إرشادات الطلب", "")}

المستندات المطلوبة:
{sections.get("المستدات المطلوبة (أصل وليس صورة)", "")}

التوقيت المحدد لإنهاء الخدمة:
{sections.get("التوقيت المحدد لإنهاء الخدمة", "")}

الرسوم المقررة للطلب:
{sections.get("الرسوم المقررة للطلب", "")}

رسوم الخدمة الإلكترونية:
{sections.get("رسوم الخدمة الإلكترونية", "")}
""".strip()

        return {
            "service_name": service_name,
            "description": service["description"],
            "instructions": sections.get(
                "إرشادات الطلب",
                ""
            ),
            "required_documents": sections.get(
                "المستدات المطلوبة (أصل وليس صورة)",
                ""
            ),
            "duration": sections.get(
                "التوقيت المحدد لإنهاء الخدمة",
                ""
            ),
            "fees": sections.get(
                "الرسوم المقررة للطلب",
                ""
            ),
            "electronic_fees": sections.get(
                "رسوم الخدمة الإلكترونية",
                ""
            ),
            "url": service["service_url"],
            "source": service["source"],
            "category": service["category"],
            "clean_text": clean_text
        }

    except Exception as e:

        print(
            "MOI Error:",
            service["service_name"],
            e
        )

        return None

### RUN ALL SOURCES

In [18]:
def get_moi_services_ajax():

    url = (
        MOI_BASE_URL +
        "/UserApplications/Services/SearchCategories"
    )

    response = session.get(
        url,
        params={
            "searching": None,
            "CategoriesParentID": 0
        },
        timeout=30
    )

    print("MOI AJAX status:", response.status_code)

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    services = []

    for card in soup.select(".portfolio-item"):

        title_tag = card.select_one(
            "h3.item-title[style='display:flex']"
        )

        description_tag = card.select_one(
            "p[data-filter]"
        )

        service_link = card.select_one(
            "a.btn.btn-primary[href*='Instructions?CategoryId=']"
        )

        instruction_button = card.select_one(
            "button[onclick*='OpenInstructionModel']"
        )

        if not title_tag:
            continue

        service_name = title_tag.get_text(
            " ",
            strip=True
        )

        description = ""

        if description_tag:
            description = description_tag.get_text(
                " ",
                strip=True
            )

        service_url = None

        if service_link:
            service_url = urljoin(
                MOI_BASE_URL,
                service_link["href"]
            )

        instruction_id = None

        if instruction_button:

            onclick = instruction_button.get(
                "onclick",
                ""
            )

            match = re.search(
                r"OpenInstructionModel\('([^']+)'",
                onclick
            )

            if match:
                instruction_id = match.group(1)

        if not service_url:
            continue

        services.append({
            "service_name": service_name,
            "description": description,
            "service_url": service_url,
            "instruction_id": instruction_id,
            "source": "وزارة الداخلية - الجوازات والهجرة والجنسية",
            "category": "Government Services"
        })

    return services

In [19]:
def run_parallel(func, items, max_workers, label):

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = [executor.submit(func, item) for item in items]

        for i, future in enumerate(as_completed(futures), start=1):

            result = future.result()

            if result:
                results.append(result)

            if i % 10 == 0 or i == len(items):
                print(f"{label}: {i}/{len(items)}")

    return results


start = time.time()

all_documents = []

khadamat_links = get_khadamat_service_links()
moi_services = get_moi_services_ajax()

print("Found Khadamat links:", len(khadamat_links))
print("Found MOI services:", len(moi_services))

with ThreadPoolExecutor(max_workers=3) as top:

    f_khadamat = top.submit(scrape_khadamat_all, khadamat_links, 10)
    f_msit = top.submit(run_parallel, scrape_msit_service, MSIT_SERVICES, 5, "MSIT")
    f_moi = top.submit(run_parallel, scrape_moi_service, moi_services, 8, "MOI")

    khadamat_documents = f_khadamat.result()
    msit_documents = f_msit.result()
    moi_documents = f_moi.result()

all_documents.extend(khadamat_documents)
all_documents.extend(msit_documents)
all_documents.extend(moi_documents)

# FINAL SUMMARY
print("\n" + "=" * 70)
print("FINAL DATASET")
print("Khadamat:", len(khadamat_documents))
print("MSIT:", len(msit_documents))
print("MOI:", len(moi_documents))
print("TOTAL:", len(all_documents))
print(f"Time: {time.time() - start:.1f}s")

Khadamat Misr status: 200
MOI AJAX status: 200
Found Khadamat links: 133
Found MOI services: 35
MOI: 10/35
MOI: 20/35
Khadamat: 25/133
MOI: 30/35
Khadamat: 50/133
Khadamat: 75/133
Khadamat: 100/133
MOI: 35/35
Khadamat: 125/133
Skipped - request failed: دورة استخراج بدل فاقد للبطاقة الذكية
Skipped - request failed: أماكن تنشيط البطاقات الذكية
Skipped - request failed: التقديم على مخبز تمويني
Skipped - request failed: دورة استخراج بطاقة تموينية ذكية جديدة
Skipped - request failed: التقديم على منفذ تموينى
MSIT: 5/5

FINAL DATASET
Khadamat: 130
MSIT: 0
MOI: 35
TOTAL: 165
Time: 9.2s


## Chunking 

In [20]:
langchain_documents = []

for doc in all_documents:

    langchain_documents.append(
        Document(
            page_content=doc["clean_text"],
            metadata={
                "service_name": doc["service_name"],
                "url": doc["url"],
                "source": doc.get("source", "")
            }
        )
    )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(langchain_documents)

# put the service name at the start of every chunk
for c in chunks:
    if not c.page_content.startswith("اسم الخدمة"):
        c.page_content = (
            f"اسم الخدمة: {c.metadata['service_name']}\n"
            f"{c.page_content}"
        )

print("LangChain Documents:", len(langchain_documents))
print("Chunks:", len(chunks))

LangChain Documents: 165
Chunks: 206


## Embedding

In [21]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [22]:
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS vector store created successfully")

FAISS vector store created successfully


## Search

In [23]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

## Try - Question

In [24]:
prompt_template = """
أنت مساعد لخدمات المواطنين في مصر. أجب عن سؤال المستخدم اعتمادًا على السياق فقط، باللهجة المصرية البسيطة، ولا تضف أي معلومة غير موجودة في السياق.
استخدم الخدمة المطابقة لسؤال المستخدم فقط، وأجب عن المعلومة التي يسأل عنها بالتحديد (المستندات، الرسوم، المدة، الخطوات، الشروط، أو وصف الخدمة).
إذا كانت الإجابة عدة بنود فاذكرها كلها كاملة، كل بند في سطر يبدأ بـ "-".
حافظ على أسماء المستندات والأرقام والمبالغ كما وردت في السياق.
إذا لم توجد الإجابة في السياق اكتب: المعلومة غير موجودة في المصادر المتاحة.

مثال 1 (للشكل فقط):
السياق:
اسم الخدمة: تجديد رخصة قيادة
المستندات المطلوبة:
- الرخصة القديمة
- شهادة طبية
- صورة البطاقة
السؤال:
عايز اجدد رخصة القيادة محتاج ايه؟
الإجابة:
علشان تجدد رخصة القيادة هتحتاج:
- الرخصة القديمة
- شهادة طبية
- صورة البطاقة

مثال 2 (للشكل فقط):
السياق:
اسم الخدمة: استخراج شهادة قيد
الرسوم المقررة للطلب:
100 جنيه
التوقيت المحدد لإنهاء الخدمة:
خلال 5 أيام
السؤال:
الخدمة دي بكام وبتخلص امتى؟
الإجابة:
رسوم الخدمة 100 جنيه، وبتخلص خلال 5 أيام.

الآن أجب عن السؤال التالي بنفس الطريقة.

السياق:
{context}

السؤال:
{question}

الإجابة:
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [25]:
def ask(question, k=4):

    results = retriever.invoke(question)

    context = "\n\n---\n\n".join(
        doc.page_content
        for doc in results
    )

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    answer = generate_text(final_prompt)

    return answer, results

In [26]:
# Try 
question = "انا عايز اطلع بطاقة رقم قومي اول مره"

answer, sources = ask(question)

print(answer)

علشان تطلع بطاقة رقم قومي أول مرة هتحتاج:
- طالب: شهادة ميلاد مختومة من جهة الدراسة
- ضامن: أقارب حتى الدرجة الثالثة


## Ngrok 

In [27]:
app = Flask(__name__)
gpu_lock = threading.Lock() 


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})


@app.route("/ask", methods=["POST"])
def ask_route():
    try:
        data = request.get_json()
        question = data.get("question", "").strip()
        k = int(data.get("k", 4))
        max_new_tokens = int(data.get("max_new_tokens", 300))

        if not question:
            return jsonify({"error": "empty question"}), 400

        docs = vectorstore.similarity_search(question, k=k)
        context = "\n\n---\n\n".join(d.page_content for d in docs)
        final_prompt = prompt.format(context=context, question=question)

        with gpu_lock:
            answer = generate_text(final_prompt, max_new_tokens=max_new_tokens)

        sources, seen = [], set()
        for d in docs:
            if d.metadata["url"] not in seen:
                seen.add(d.metadata["url"])
                sources.append({
                    "service_name": d.metadata["service_name"],
                    "url": d.metadata["url"],
                    "source": d.metadata.get("source", ""),
                })

        return jsonify({"answer": answer, "sources": sources})

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        print(tb)
        return jsonify({"error": str(e), "traceback": tb}), 50

In [28]:
def run_server():
    app.run(host="0.0.0.0", port=8000, threaded=True)


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(5)
print("Thread alive?", server_thread.is_alive())

ngrok.kill()
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
print("Public URL:", public_url.public_url)  

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.19.2.2:8000
INFO:werkzeug:Press CTRL+C to quit


Thread alive? True
Public URL: https://stinging-gruffly-progress.ngrok-free.dev                                        


# Thank You